<a href="https://colab.research.google.com/github/pranathi0104/Day-trip-agent-/blob/main/Copy_of_ADK_Learning_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Welcome to Your ADK Adventure - Tools & Memory! 🚀

Welcome, Agent Architect! This notebook is your guide to giving your AI agents two essential superpowers: custom tools and conversational memory.

By the end of this adventure, you will be able to:

- **Build a Foundational Agent**: Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).

- **Grant New Skills with Custom Tools**: Teach an agent to perform new tasks by connecting it to external APIs, like a real-time weather service.

- **Create a Team of Agents**: Assemble a multi-agent system where a primary agent can delegate specialized tasks to other agents.

- **Master Conversational Memory**: Understand the critical role of Sessions in enabling agents to remember previous interactions, handle feedback, and carry on a coherent conversation.


Let's get this adventure started!

## Author

HI, I'm Qingyue (Annie) Wang, a developer advocate and AI engineer at **Google**, passionate about helping developers build with AI and cloud technologies :)


If you have questions with this notebook, contact me on [LinkedIn](https://www.linkedin.com/in/anniewangtech/) , [X](https://twitter.com/anniewangtech) or email anniewangtech0510@Gmail.com


```
  (\__/)
  (•ㅅ•)
  /づ  📚      Enjoy learning AI Agents :)
```


-------------
### 🎁 🛑 Important Prerequisite: Setup Your Environment! 🛑 🎁
-----------------------------------------------------------------------------

👉 **Get Your API Key HERE**: https://codelabs.developers.google.com/onramp/instructions#1

 -----------------------------------------------------------------------------

```
 ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️
   /\_/\     /\_/\     /\_/\      /\_/\       /\_/\
  ( ^_^ )   ( -.- )   ( >_< )   ( =^.^= )    ( o_o )             
```


## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [1]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
import vertexai
from google.colab import auth
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part


print("✅ All libraries are ready to go!")


✅ All libraries are ready to go!


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


### Authenticate and Configure Your Project
To use Vertex AI, you need an active Google Cloud project. This section handles authenticating your environment and setting up the necessary project configurations.

In [3]:
# ---  Authentication & Project Configuration ---

# Authenticate user in Colab
if "google.colab" in sys.modules:
    auth.authenticate_user()
    print("✅ Authenticated successfully.")

✅ Authenticated successfully.


In [4]:
# @title Set Your Google Cloud Project Details
PROJECT_ID = "core-catalyst-496511-k9"             # @param {type:"string"}
LOCATION = "us-central1"               # @param {type:"string"}

# Set environment variables for the ADK and gcloud
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}

print(f"\n✅ Vertex AI configured for project '{PROJECT_ID}' in '{LOCATION}'.")


✅ Vertex AI configured for project 'core-catalyst-496511-k9' in 'us-central1'.


---
## Part 1: Your First Agent - The Day Trip Genie 🧞

Meet your first creation! The `day_trip_agent` is a simple but powerful assistant. We're making it a little smarter by teaching it to understand **budget constraints**.

* **Agent**: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
* **Session**: The conversation history. For this simple agent, it's just a container for a single request-response.
* **Runner**: The engine that connects the `Agent` and the `Session` to process your request and get a response.

```
+--------------------------------------------------+
|         Spontaneous Day Trip Agent 🤖            |
|--------------------------------------------------|
|  Model: gemini-2.5-flash                         |
|  Description:                                    |
|   Generates full-day trip itineraries based on   |
|   mood, interests, and budget                    |
|--------------------------------------------------|
|  🔧 Tools:                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capabilities:                                |
|   - Budget Awareness (cheap / splurge)           |
|   - Mood Matching (adventurous, relaxing, etc.)  |
|   - Real-Time Info (hours, events)               |
|   - Morning / Afternoon / Evening plan           |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    |   User Input     |
    |------------------|
    |  Mood            |
    |  Interests       |
    |  Budget          |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|             Output: Markdown Itinerary           |
|--------------------------------------------------|
| - Time blocks (Morning / Afternoon / Evening)    |
| - Venue names with links and hours               |
| - Budget-matching activities                     |
+--------------------------------------------------+
```


In [5]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [6]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [7]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!'

🚀 Running query for agent: 'day_trip_agent' in session: '981b8947-f3a9-42c7-9b45-5fc76b680ead'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Escape to a day of tranquility and artistic inspiration with this affordable day trip from Sunnyvale, exploring the charming city of Palo Alto and the renowned Stanford University campus.

## Relaxing & Artsy Day Trip: Palo Alto & Stanford

### Morning: A Serene Start at Elizabeth F. Gamble Garden (9:30 AM - 12:00 PM)

Begin your day with a visit to the **Elizabeth F. Gamble Garden** in Palo Alto. This historic and beautifully maintained property offers a serene escape, perfect for unwinding. Stroll through various themed gardens, including a rose garden, cutting garden, formal herb garden, and a wisteria garden. The garden is a non-profit organization and is open to the public for free during daylight hours,

Escape to a day of tranquility and artistic inspiration with this affordable day trip from Sunnyvale, exploring the charming city of Palo Alto and the renowned Stanford University campus.

## Relaxing & Artsy Day Trip: Palo Alto & Stanford

### Morning: A Serene Start at Elizabeth F. Gamble Garden (9:30 AM - 12:00 PM)

Begin your day with a visit to the **Elizabeth F. Gamble Garden** in Palo Alto. This historic and beautifully maintained property offers a serene escape, perfect for unwinding. Stroll through various themed gardens, including a rose garden, cutting garden, formal herb garden, and a wisteria garden. The garden is a non-profit organization and is open to the public for free during daylight hours, making it an ideal budget-friendly option for relaxation and connecting with nature. It's described as a "center of calm" in the middle of the city, offering a chance to "relax and re-center."

### Lunch: Affordable Delights in Palo Alto (12:30 PM - 1:30 PM)

For an affordable and flavorful lunch, head to one of Palo Alto's well-regarded casual eateries. Consider **Oren's Hummus Shop** or **Mediterranean Wraps**. Oren's Hummus offers satisfying hummus plates, shawarma wraps, and falafel bowls that are both delicious and budget-friendly. Alternatively, Mediterranean Wraps is known for its tasty and filling options.

### Afternoon: Artistic Immersion at Stanford University (2:00 PM - 5:30 PM)

Immerse yourself in art and culture on the stunning Stanford University campus.

*   **Rodin Sculpture Garden:** Start your artistic journey outdoors at the **Rodin Sculpture Garden**, located adjacent to the Cantor Arts Center. This impressive outdoor collection features one of the largest gatherings of Auguste Rodin's bronzes outside of Paris, including iconic works like "The Gates of Hell" and "The Thinker." The garden is open 24/7, and admission is free.
*   **Cantor Arts Center:** Step inside the **Cantor Arts Center**, where admission is always free. The museum houses diverse collections spanning 4,000 years of art history, with a wide range of changing exhibitions. On Sundays, the museum is open from 10:00 AM to 5:00 PM. Parking on the Stanford campus is free on weekends, making your visit even more affordable.

### Evening: Casual Dinner & Reflection (6:00 PM onwards)

Conclude your relaxing and artsy day with a casual and affordable dinner. You could return to downtown Palo Alto and try **Zareen's**, highly praised for its affordable and delicious Pakistani/Indian cuisine. Another option for classic American comfort food is **Palo Alto Creamery**, known for its burgers and milkshakes. Enjoy a relaxed meal as you reflect on the beauty and art encountered throughout your day.

--------------------------------------------------



---
## Part 2: Supercharging Agents with Custom Tools 🛠️

So far, we've used the powerful built-in `GoogleSearch` tool. But the true power of agents comes from connecting them to your own logic and data sources.

This is where **custom tools** come in. Let's explore three patterns for giving your agent new skills, using real-world, practical examples.

### 2.1 The Simple `FunctionTool`: Calling a Real-Time Weather API

The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.

**Key Concept:** The function's **docstring** is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose, parameters, and when to use it.

In this example, we'll create a tool that calls the **free, public U.S. National Weather Service API** to get a real-time forecast. No API key needed!

In [8]:
# --- Tool Definition: A function that calls a live public API ---
import requests
import json

# A simple lookup to avoid needing a separate geocoding API for this example
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324"
}

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in the US.

    Args:
        location: The city name, e.g., "San Francisco".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # NWS API requires 2 steps: 1. Get the forecast URL from the coordinates.
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status() # Raise an exception for bad status codes
        forecast_url = points_response.json()['properties']['forecast']

        # 2. Get the actual forecast from the URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Extract the relevant forecast details
        current_period = forecast_response.json()['properties']['periods'][0]
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}

# --- Agent Definition: An agent that USES the new tool ---

weather_agent = Agent(
    name="weather_aware_planner",
    model="gemini-2.5-flash",
    description="A trip planner that checks the real-time weather before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools=[get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")

🌦️ Agent 'weather_aware_planner' is created and can now call a live weather API!


In [9]:
# --- Let's test the Weather-Aware Planner ---

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    query = "I want to go hiking near Lake Tahoe, what's the weather like?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()

🗣️ User Query: 'I want to go hiking near Lake Tahoe, what's the weather like?'

🚀 Running query for agent: 'weather_aware_planner' in session: '749cc403-8495-4254-b23f-60baa65115c3'...


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'location': 'South Lake Tahoe'
        },
        id='adk-80a74b89-16d2-45ef-830e-e8ccc24d5f93',
        name='get_live_weather_forecast'
      ),
      thought_signature=b"\n\xf1\x03\x01\x8f=k_z\xa2`\x1cl\xb0*\xb6\x9f\x14\xc7\xe0\xd1\x03\xd9\xefj\x0f\xae>\xec\x8a\xc2\xa3\x963O\x18\x95J\xef*1\xab\xb1QX\xf6'r\x89n\xa7\xfbtF\xb3F\xd9\x9d\xd5\x928\xf1KCc\xab6b\xd3\x03R\xafX\xec\xd3$\x9ek]d\xdax\x08\xf4\r \xc8\x02r 8\x98\xc1q\x08\xf4:...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=11,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=11
    ),
  ],
  prompt_to

The weather near Lake Tahoe (South Lake Tahoe) is partly cloudy with a low around 39°F, and a north wind of 0 to 10 mph. It might be a bit chilly for hiking, so I recommend dressing in layers and checking trail conditions as some higher elevation trails might have snow or ice.

--------------------------------------------------



## 2.2 The Agent-as-a-Tool: Consulting a Specialist 🧑‍🍳

Why build one agent that does everything when you can build a **team of specialist agents?** The **Agent-as-a-Tool** pattern allows one agent to delegate a task to another agent.

**Key Concept:** This is different from a sub-agent. When Agent A calls Agent B as a tool, Agent B's response is passed **back to Agent A**. Agent A then uses that information to form its own final response to the user. It's a powerful way to compose complex behaviors from simpler, focused, and reusable agents.

### How It Works

Our top-level agent, the `trip_data_concierge_agent`, acts as the **Orchestrator**. It has two tools at its disposal:

1.  `call_db_agent`: A function that internally calls our `db_agent` to fetch raw data.
2.  `call_concierge_agent`: A function that calls the `concierge_agent`.

The `concierge_agent` itself has a tool: the `food_critic_agent`.

The flow for a complex query is:

1.  **User** asks the `trip_data_concierge_agent` for a hotel and a nearby restaurant.
2.  The **Orchestrator** first calls `call_db_agent` to get hotel data.
3.  The data is saved in `tool_context.state`.
4.  The **Orchestrator** then calls `call_concierge_agent`, which retrieves the hotel data from the context.
5.  The `concierge_agent` receives the request and decides it needs to use its own tool, the `food_critic_agent`.
6.  The `food_critic_agent` provides a witty recommendation.
7.  The `concierge_agent` gets the critic's response and politely formats it.
8.  This final, polished response is returned to the **Orchestrator**, which presents it to the user.

                         +-----------------------------------------------------------+
                         |              🧭 Trip Data Concierge Agent                 |
                         |-----------------------------------------------------------|
                         |  Model: gemini-2.5-flash                                  |
                         |  Description:                                             |
                         |   Orchestrates database query and travel recommendation  |
                         |-----------------------------------------------------------|
                         |  🔧 Tools:                                                |
                         |   1. call_db_agent                                        |
                         |   2. call_concierge_agent                                 |
                         +-----------------------------------------------------------+
                                      /                                \
                                     /                                  \
                                    ▼                                    ▼
        +-------------------------------------------+    +---------------------------------------------+
        |            🔧 Tool: call_db_agent         |    |         🔧 Tool: call_concierge_agent        |
        |-------------------------------------------|    |---------------------------------------------|
        | Calls: db_agent                           |    | Calls: concierge_agent                       |
        |                                           |    | Uses data from db_agent for recommendations |
        +-------------------------------------------+    +---------------------------------------------+
                                |                                          |
                                ▼                                          ▼
       +--------------------------------------------+   +------------------------------------------------+
       |              📦 db_agent                   |   |             🤵 concierge_agent                  |
       |--------------------------------------------|   |------------------------------------------------|
       | Model: gemini-2.5-flash                    |   | Model: gemini-2.5-flash                         |
       | Role: Return mock JSON hotel data          |   | Role: Hotel staff that handles user Q&A        |
       +--------------------------------------------+   | Tools:                                          |
                                                         |  - food_critic_agent                           |
                                                         +------------------------------------------------+
                                                                                 |
                                                                                 ▼
                                                       +------------------------------------------------+
                                                       |          🍽️ food_critic_agent                  |
                                                       |------------------------------------------------|
                                                       | Model: gemini-2.5-flash                         |
                                                       | Role: Gives a witty restaurant recommendation   |
                                                       +------------------------------------------------+


In [10]:
import asyncio
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool import AgentTool

# Assume 'db_agent' is a pre-defined NL2SQL Agent
# For this example, we'll create placeholder agents.

db_agent = Agent(
    name="db_agent",
    model="gemini-2.5-flash",
    instruction="You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}")

# --- 1. Define the Specialist Agents ---

# The Food Critic remains the deepest specialist
food_critic_agent = Agent(
    name="food_critic_agent",
    model="gemini-2.5-flash",
    instruction="You are a snobby but brilliant food critic. You ONLY respond with a single, witty restaurant suggestion near the provided location.",
)

# The Concierge knows how to use the Food Critic
concierge_agent = Agent(
    name="concierge_agent",
    model="gemini-2.5-flash",
    instruction="You are a five-star hotel concierge. If the user asks for a restaurant recommendation, you MUST use the `food_critic_agent` tool. Present the opinion to the user politely.",
    tools=[AgentTool(agent=food_critic_agent)]
)


# --- 2. Define the Tools for the Orchestrator ---

async def call_db_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    Use this tool FIRST to connect to the database and retrieve a list of places, like hotels or landmarks.
    """
    print("--- TOOL CALL: call_db_agent ---")
    agent_tool = AgentTool(agent=db_agent)
    db_agent_output = await agent_tool.run_async(
        args={"request": question}, tool_context=tool_context
    )
    # Store the retrieved data in the context's state
    tool_context.state["retrieved_data"] = db_agent_output
    return db_agent_output


async def call_concierge_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    After getting data with call_db_agent, use this tool to get travel advice, opinions, or recommendations.
    """
    print("--- TOOL CALL: call_concierge_agent ---")
    # Retrieve the data fetched by the previous tool
    input_data = tool_context.state.get("retrieved_data", "No data found.")

    # Formulate a new prompt for the concierge, giving it the data context
    question_with_data = f"""
    Context: The database returned the following data: {input_data}

    User's Request: {question}
    """

    agent_tool = AgentTool(agent=concierge_agent)
    concierge_output = await agent_tool.run_async(
        args={"request": question_with_data}, tool_context=tool_context
    )
    return concierge_output


# --- 3. Define the Top-Level Orchestrator Agent ---

trip_data_concierge_agent = Agent(
    name="trip_data_concierge",
    model="gemini-2.5-flash",
    description="Top-level agent that queries a database for travel data, then calls a concierge agent for recommendations.",
    tools=[call_db_agent, call_concierge_agent],
    instruction="""
    You are a master travel planner who uses data to make recommendations.

    1.  **ALWAYS start with the `call_db_agent` tool** to fetch a list of places (like hotels) that match the user's criteria.

    2.  After you have the data, **use the `call_concierge_agent` tool** to answer any follow-up questions for recommendations, opinions, or advice related to the data you just found.
    """,
)

print(f"✅ Orchestrator Agent '{trip_data_concierge_agent.name}' is defined and ready.")

✅ Orchestrator Agent 'trip_data_concierge' is defined and ready.


In [11]:
# --- Let's test the Trip Data Concierge Agent ---

async def run_trip_data_concierge():
    """
    Sets up a session and runs a query against the top-level
    trip_data_concierge_agent.
    """
    # Create a new, single-use session for this query
    concierge_session = await session_service.create_session(
        app_name=trip_data_concierge_agent.name,
        user_id=my_user_id
    )

    # This query is specifically designed to trigger the full two-step process:
    # 1. Get data from the db_agent.
    # 2. Get a recommendation from the concierge_agent based on that data.
    query = "Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews."
    print(f"🗣️ User Query: '{query}'")

    # We call our existing helper function with the top-level orchestrator agent
    await run_agent_query(trip_data_concierge_agent, query, concierge_session, my_user_id)

# Run the test
await run_trip_data_concierge()

🗣️ User Query: 'Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews.'

🚀 Running query for agent: 'trip_data_concierge' in session: 'fb2d70fe-fb99-4775-80fa-5a031d1cb4bc'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'top-rated hotels in San Francisco'
        },
        id='adk-6c058f55-49cf-426a-8c26-4ddae479e3da',
        name='call_db_agent'
      ),
      thought_signature=b'\n\x99\x02\x01\x8f=k_\x9c\xcd\x1cN\xcd\xb1}\xff\xc7\xa5\x02,o\x1f[\xbf\xbfbFaQ\x1c\x7f\xa3\xf1\x00\xac\x8b\x87\x0f+\xfbK\x86\x93N6\xda\x87\xb4\xca\xe43\xc0\xcb\xac\xf8\xfa\x01\xe2\xd3]\x18O\xbcx\xcdn;\x89\\\xcd-\xec\xaa\xf4l\xf6\xbc\xbc/w\x00\x91u\x9dAZ\x84\xb0\x9b\xe4\xf8\x8d\x9aqV\xc9\x1c...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None e

Certainly! Among the top-rated hotels in San Francisco, Seaside Inn has the most reviews.

For a dinner spot near Seaside Inn, I recommend Che Fico. It's known to be a place where the *cognoscenti* congregate and the pasta is said to be excellent.

--------------------------------------------------



---
## Part 3: Agent with a Memory - The Adaptive Planner 🗺️

Now, let's see an agent that not only **remembers** but also **adapts**. We'll challenge the `multi_day_trip_agent` to re-plan part of its itinerary based on our feedback. This is a much more realistic test of conversational AI.

```
+-----------------------------------------------------+
|         Adaptive Multi-Day Trip Agent 🗺️           |
|-----------------------------------------------------|
|  Model: gemini-2.5-flash                            |
|  Description:                                       |
|   Builds multi-day travel itineraries step-by-step, |
|   remembers previous days, adapts to feedback       |
|-----------------------------------------------------|
|  🔧 Tools:                                          |
|   - Google Search                                   |
|-----------------------------------------------------|
|  🧠 Capabilities:                                   |
|   - Memory of past conversation & preferences       |
|   - Progressive planning (1 day at a time)          |
|   - Adapts to user feedback                         |
|   - Ensures activity variety across days            |
+-----------------------------------------------------+

            ▲
            |
    +---------------------------+
    |     User Interaction      |
    |---------------------------|
    | - Destination             |
    | - Trip duration           |
    | - Interests & feedback    |
    +---------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Day-by-Day Itinerary Generation              |
|-----------------------------------------------------|
|  🗓️ Day N Output (Markdown format):                 |
|   - Morning / Afternoon / Evening activities        |
|   - Personalized & context-aware                    |
|   - Changes accepted, feedback acknowledged         |
+-----------------------------------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Next Day Planning Triggered 🚀               |
|-----------------------------------------------------|
| - Builds on prior days                              |
| - Avoids repetition                                 |
| - Asks user for confirmation before proceeding      |
+-----------------------------------------------------+
```


In [12]:
# --- Agent Definition: The Adaptive Planner ---

def create_multi_day_trip_agent():
    """Create the Progressive Multi-Day Trip Planner agent"""
    return Agent(
        name="multi_day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction="""
        You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.

        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        """,
        tools=[google_search]
    )

multi_day_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_agent.name}' is created and ready to plan and adapt!")

🗺️ Agent 'multi_day_trip_agent' is created and ready to plan and adapt!


### Scenario 3a: Agent WITH Memory (Using a SINGLE Session) ✅

First, let's see the correct way to do it. We will use the **exact same `trip_session` object** for the entire conversation. Watch how the agent remembers the context from Turn 1 to correctly handle the requests in Turn 2 and 3.

In [13]:
# --- Scenario 2: Testing Adaptation and Memory ---

async def run_adaptive_memory_demonstration():
    print("### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###")

    # Create ONE session that we will reuse for the whole conversation
    trip_session = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"Created a single session for our trip: {trip_session.id}")

    # --- Turn 1: The user initiates the trip ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    print(f"\n🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, trip_session, my_user_id)

    # --- Turn 2: The user gives FEEDBACK and asks for a CHANGE ---
    # We use the EXACT SAME `trip_session` object!
    query2 = "That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?"
    print(f"\n🗣️ User (Turn 2 - Feedback): '{query2}'")
    await run_agent_query(multi_day_agent, query2, trip_session, my_user_id)

    # --- Turn 3: The user confirms and asks to continue ---
    query3 = "Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind."
    print(f"\n🗣️ User (Turn 3 - Confirmation): '{query3}'")
    await run_agent_query(multi_day_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()

### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###
Created a single session for our trip: 1a1f491d-56f5-48e8-83b2-3c218cdc0f7b

🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '1a1f491d-56f5-48e8-83b2-3c218cdc0f7b'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Sounds like a fantastic trip! Lisbon is perfect for historic sites and delicious local food. I'll plan your itinerary day by day.

Here's a proposal for **Day 1**:

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama & São Jorge Castle**
    *   Start your day by wandering through the narrow, winding streets of **Alfama**, Lisbon's oldest district. Immerse yourself in its medieval charm and discover hidden viewpoints.
    *   Head up to **São Jorge Castle (Castelo de São Jorge)**, a historic fortification offering panoramic views of the city an

Sounds like a fantastic trip! Lisbon is perfect for historic sites and delicious local food. I'll plan your itinerary day by day.

Here's a proposal for **Day 1**:

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama & São Jorge Castle**
    *   Start your day by wandering through the narrow, winding streets of **Alfama**, Lisbon's oldest district. Immerse yourself in its medieval charm and discover hidden viewpoints.
    *   Head up to **São Jorge Castle (Castelo de São Jorge)**, a historic fortification offering panoramic views of the city and the Tagus River. Explore its ramparts, gardens, and archaeological site.

*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Lunch in Alfama**
    *   Enjoy an authentic Portuguese meal at a traditional "tasca" (a small, informal restaurant) within the Alfama district. Try some grilled sardines or a bacalhau (codfish) dish.

*   **Afternoon (2:30 PM - 6:00 PM): Lisbon Cathedral & Baixa District**
    *   Walk to the **Lisbon Cathedral (Sé de Lisboa)**, the city's oldest church, showcasing a mix of Romanesque, Gothic, and Baroque styles.
    *   Continue to the **Baixa district**, a grid-like area rebuilt after the 1755 earthquake. Admire its elegant squares like Praça do Comércio and Rossio Square.

*   **Evening (7:00 PM onwards): Dinner with Fado Music**
    *   Experience a traditional Portuguese dinner in Alfama or Bairro Alto, where you can often find restaurants offering live **Fado music**, Portugal's soulful and melancholic musical genre.

How does this sound for your first day in Lisbon?

--------------------------------------------------


🗣️ User (Turn 2 - Feedback): 'That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '1a1f491d-56f5-48e8-83b2-3c218cdc0f7b'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Okay, I understand! No problem, we can definitely swap out the castle for another great historical site.

How about we replace São Jorge Castle with a visit to the **National Pantheon**? It's a magnificent historical building, originally a church, that now houses the tombs of important Portuguese personalities. It offers stunning architecture and a different kind of historical experience, still providing great views from its dome, similar to the castle but without being a fortification.

Here's the revised **Day 1** plan:

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama & 

Okay, I understand! No problem, we can definitely swap out the castle for another great historical site.

How about we replace São Jorge Castle with a visit to the **National Pantheon**? It's a magnificent historical building, originally a church, that now houses the tombs of important Portuguese personalities. It offers stunning architecture and a different kind of historical experience, still providing great views from its dome, similar to the castle but without being a fortification.

Here's the revised **Day 1** plan:

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama & National Pantheon**
    *   Start your day by wandering through the narrow, winding streets of **Alfama**, Lisbon's oldest district. Immerse yourself in its medieval charm and discover hidden viewpoints.
    *   Head to the **National Pantheon (Panteão Nacional)**, a grand 17th-century building originally consecrated as the Church of Santa Engrácia. It houses the tombs of many significant Portuguese figures and offers impressive architecture and panoramic views from its terrace.

*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Lunch in Alfama**
    *   Enjoy an authentic Portuguese meal at a traditional "tasca" (a small, informal restaurant) within the Alfama district. Try some grilled sardines or a bacalhau (codfish) dish.

*   **Afternoon (2:30 PM - 6:00 PM): Lisbon Cathedral & Baixa District**
    *   Walk to the **Lisbon Cathedral (Sé de Lisboa)**, the city's oldest church, showcasing a mix of Romanesque, Gothic, and Baroque styles.
    *   Continue to the **Baixa district**, a grid-like area rebuilt after the 1755 earthquake. Admire its elegant squares like Praça do Comércio and Rossio Square.

*   **Evening (7:00 PM onwards): Dinner with Fado Music**
    *   Experience a traditional Portuguese dinner in Alfama or Bairro Alto, where you can often find restaurants offering live **Fado music**, Portugal's soulful and melancholic musical genre.

How does this revised first day sound to you?

--------------------------------------------------


🗣️ User (Turn 3 - Confirmation): 'Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '1a1f491d-56f5-48e8-83b2-3c218cdc0f7b'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Fantastic! I'm glad Day 1 is to your liking. Let's move on to an exciting **Day 2**, focusing on more historical exploration and, of course, incredible local food experiences.

Here's a proposal for **Day 2**:

*   **Morning (9:00 AM - 1:00 PM): Belém's Age of Discoveries**
    *   Begin your day in the historic **Belém district**, which played a pivotal role in Portugal's Age of Discoveries.
    *   Visit the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, a UNESCO World Heritage site and a masterpiece of Manueline architecture. Explore its ornate cloisters and the church where explorer Vasco da Gam

Fantastic! I'm glad Day 1 is to your liking. Let's move on to an exciting **Day 2**, focusing on more historical exploration and, of course, incredible local food experiences.

Here's a proposal for **Day 2**:

*   **Morning (9:00 AM - 1:00 PM): Belém's Age of Discoveries**
    *   Begin your day in the historic **Belém district**, which played a pivotal role in Portugal's Age of Discoveries.
    *   Visit the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, a UNESCO World Heritage site and a masterpiece of Manueline architecture. Explore its ornate cloisters and the church where explorer Vasco da Gama is buried. It's recommended to arrive early to avoid crowds.
    *   Stroll along the waterfront to see the **Monument to the Discoveries (Padrão dos Descobrimentos)**, a grand tribute to Portugal's explorers. You can climb to the top for panoramic views.
    *   Admire the iconic **Belém Tower (Torre de Belém)**, a fortified tower that once guarded the entrance to Lisbon's harbor. While currently closed for restoration with reopening expected in June 2026, you can still appreciate its exterior architecture and historical significance.

*   **Lunch (1:00 PM - 2:30 PM): Taste the Original Pastéis de Belém**
    *   No visit to Belém is complete without trying the world-famous **Pastéis de Belém**. Head to the original factory, Fábrica dos Pastéis de Belém, for warm, custard-filled pastries, often enjoyed with a sprinkle of cinnamon and powdered sugar. The recipe has been a closely guarded secret since 1837.
    *   You can also find various lunch options in the Belém area.

*   **Afternoon (2:30 PM - 6:00 PM): Scenic Tram Ride & Panoramic Views**
    *   Experience the charm of Lisbon with a ride on the iconic **Tram 28**. This vintage yellow tram rattles through several historic neighborhoods, offering a scenic tour and a glimpse into daily Lisbon life.
    *   Hop off at one of Lisbon's famous viewpoints, or "miradouros," such as **Miradouro da Senhora do Monte** or **Miradouro da Graça**, for breathtaking panoramic views of the city, the castle, and the Tagus River.

*   **Evening (7:00 PM onwards): Culinary Exploration at Time Out Market**
    *   For your final evening, head to the **Time Out Market (Mercado da Ribeira)**. This vibrant food hall brings together some of Lisbon's best chefs and restaurants under one roof, offering a wide variety of traditional Portuguese and international cuisines. It's a fantastic spot to sample different dishes, from seafood to gourmet burgers, in a lively atmosphere.

How does this sound for your second day in Lisbon?

--------------------------------------------------



### Scenario 3b: Agent WITHOUT Memory (Using SEPARATE Sessions) ❌

Now, let's see what happens if we mess up our session management. Here, we'll give the agent a case of amnesia by creating a **brand new, separate session for each turn**.

Pay close attention to the agent's response to the second query. Because it's in a new session, it has no memory of the trip to Lisbon we just discussed!

In [14]:
# --- Scenario 2b: Demonstrating Memory FAILURE ---

async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print("### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###")
    print("#"*60)

    # --- Turn 1: The user initiates the trip in the FIRST session ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    session_one = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a session for Turn 1: {session_one.id}")
    print(f"🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, session_one, my_user_id)

    # --- Turn 2: The user asks to continue... but in a completely NEW session ---
    query2 = "Yes, that looks perfect! Please plan Day 2."
    session_two = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a BRAND NEW session for Turn 2: {session_two.id}")
    print(f"🗣️ User (Turn 2): '{query2}'")
    await run_agent_query(multi_day_agent, query2, session_two, my_user_id)

await run_memory_failure_demonstration()


############################################################
### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###
############################################################

Created a session for Turn 1: 9a00f7fa-47fa-4354-8010-2512b86c4f0d
🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '9a00f7fa-47fa-4354-8010-2512b86c4f0d'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Great! Lisbon is a fantastic choice with its rich history and delicious cuisine. I can definitely help you plan a memorable 2-day trip.

Let's start with **Day 1**. Given your interest in historic sites and local food, how about this itinerary?

**Day 1: Historic Alfama & Baixa Flavors**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District.** Start your day wandering through the labyrinthine streets of Alfama, Lisbon

Great! Lisbon is a fantastic choice with its rich history and delicious cuisine. I can definitely help you plan a memorable 2-day trip.

Let's start with **Day 1**. Given your interest in historic sites and local food, how about this itinerary?

**Day 1: Historic Alfama & Baixa Flavors**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District.** Start your day wandering through the labyrinthine streets of Alfama, Lisbon's oldest district. Visit the **Lisbon Cathedral (Sé de Lisboa)** and then make your way up to **São Jorge Castle (Castelo de São Jorge)** for stunning panoramic views of the city and the Tagus River. This historic Moorish castle offers a glimpse into Lisbon's past.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese in Alfama.** Enjoy a traditional Portuguese meal in Alfama. There are many cozy, family-run restaurants (tascas) serving authentic dishes like *Bacalhau à Brás* (codfish with scrambled eggs and potatoes) or grilled sardines.
*   **Afternoon (2:30 PM - 6:00 PM): Baixa and Commerce Square.** Descend from Alfama into the Baixa district, rebuilt after the 1755 earthquake. Admire the neoclassical architecture, stroll along Rua Augusta, and pass under the **Rua Augusta Arch** leading to **Praça do Comércio (Commerce Square)**, one of Europe's largest and most magnificent squares.
*   **Evening (7:30 PM onwards): Dinner & Fado in Chiado/Bairro Alto.** Head towards the vibrant Chiado or Bairro Alto districts. Enjoy dinner at a restaurant offering contemporary Portuguese cuisine or traditional fare. For an authentic cultural experience, consider finding a venue that offers **Fado music**, Lisbon's soulful and traditional music genre, often performed in intimate settings.

How does this sound for your first day in Lisbon?

--------------------------------------------------


Created a BRAND NEW session for Turn 2: c7755489-fcaa-4823-a665-a86309ed004b
🗣️ User (Turn 2): 'Yes, that looks perfect! Please plan Day 2.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'c7755489-fcaa-4823-a665-a86309ed004b'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Okay, great! I'm glad Day 1 looks perfect. Let's plan a fascinating Day 2, focusing on more history, art, and delicious Parisian food, while exploring different charming neighborhoods.

Here's a suggested itinerary for Day 2:

### **Day 2: Medieval History, Artistic Heights & Parisian Charm**

*   **Morning (9:00 AM - 1:00 PM): Île de la Cité & Sainte-Chapelle's Stained Glass**
    *   Begin your day on the **Île de la Cité**, the historical heart of Paris. Your first stop will be the iconic **Notre Dame Cathedral**. While the interior is undergoing reconstruction, you can still admire its magnificent G

Okay, great! I'm glad Day 1 looks perfect. Let's plan a fascinating Day 2, focusing on more history, art, and delicious Parisian food, while exploring different charming neighborhoods.

Here's a suggested itinerary for Day 2:

### **Day 2: Medieval History, Artistic Heights & Parisian Charm**

*   **Morning (9:00 AM - 1:00 PM): Île de la Cité & Sainte-Chapelle's Stained Glass**
    *   Begin your day on the **Île de la Cité**, the historical heart of Paris. Your first stop will be the iconic **Notre Dame Cathedral**. While the interior is undergoing reconstruction, you can still admire its magnificent Gothic exterior and the surrounding area. The cathedral is open daily from approximately 7:50 AM to 7:00 PM.
    *   Just a short walk from Notre Dame, immerse yourself in the breathtaking beauty of **Sainte-Chapelle**. This 13th-century Gothic chapel is renowned for its stunning, kaleidoscopic stained-glass windows, which depict over 1,000 scenes from the Old and New Testaments. Sainte-Chapelle opens at 9:00 AM.

*   **Lunch (1:00 PM - 2:30 PM): Culinary Delights in the Latin Quarter**
    *   Cross into the vibrant **Latin Quarter**, a historic district known for its intellectual past and lively atmosphere. This area offers a wide variety of dining options, from traditional French bistros to diverse international cuisine, catering to all price points. You'll find many charming spots for a delicious and authentic Parisian lunch.

*   **Afternoon (2:30 PM - 6:30 PM): Artistic Montmartre & Sacré-Cœur**
    *   Head north to the picturesque neighborhood of **Montmartre**, famous for its artistic heritage and bohemian charm. Start by visiting the magnificent **Sacré-Cœur Basilica**, perched atop the highest point in Paris. Entry to the basilica is free, and its steps offer unparalleled panoramic views of the city.
    *   After exploring Sacré-Cœur, wander through **Place du Tertre**, where artists still set up their easels and paint portraits or Parisian scenes. This charming square captures the artistic spirit of Montmartre.

*   **Evening (6:30 PM onwards): Crêpes and Montmartre Ambiance**
    *   Conclude your day with a relaxed dinner in Montmartre. The neighborhood is perfect for enjoying classic French fare or indulging in some delicious crêpes. Consider a local crêperie like Crêperie Brocéliande, known for its authentic and even gluten-free options.

How does this sound for your second day in Paris?

--------------------------------------------------



See? The agent was confused! It likely asked what destination or what trip we were talking about. Because the second query was in a fresh, isolated session, the agent had no memory of planning Day 1 in Lisbon.

This perfectly illustrates why **managing sessions is the key to building truly conversational agents!**

---
## 🎉 Congratulations! 🎉

Congratulations on completing your ADK adventure into Tools and Memory! You've taken a massive leap from building single-shot agents to creating dynamic, stateful AI systems.

Let's recap the powerful concepts you've mastered:

- **Fundamental Agent & Tools**: You started by building a "Day Trip Genie" and equipped it with its first tool, GoogleSearch.

- **Custom Function Tools**: You gave your agent a new sense by creating a custom tool to fetch live data from the U.S. National Weather Service API.

- **Agent-as-a-Tool**: You orchestrated a sophisticated hierarchy where agents delegate tasks to other, more specialized agents, creating a collaborative team.

- **The Power of Memory**: Most importantly, you saw firsthand how managing a single, persistent Session allows an agent to remember context, adapt to user feedback, and conduct a meaningful, multi-turn conversation.

```
   __            /\_/\         /\_/\        /\_/\         __             (\__/)
o-''|\_____/).  ( o.o )       ( -.- )      ( ^_^ )     o-''|\_____/).    ( ^_^ )
 \_/|_)     )    > ^ <         > * <        >💖<         \_/|_)     )     / >🌸< \
    \  __  /                                              \  __  /         /   \
    (_/ (_/                                               (_/ (_/        (___|___)
```
